# Retail Demand Forecasting — Analysis

This notebook is the *narrative* behind the pipeline: why the metric is MASE and
not MAPE, why baselines are in the model set at all, and how forecast error turns
into money.

Nothing is implemented here. Every function comes from `retail_intel`, so the
notebook and the production pipeline cannot disagree — which is the failure mode
of the notebook this replaced, where the feature engineering lived only in cells
and the deployed API used something else entirely.

**Run `make seed` (or `make pipeline SOURCE=...`) first** so the warehouse is
populated.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from retail_intel.db import read_sql

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
plt.rcParams["figure.figsize"] = (11, 4.5)
pd.set_option("display.width", 200)

## 1. The data after cleaning

In [ ]:
tx = read_sql("SELECT * FROM transactions")
tx["invoice_date"] = pd.to_datetime(tx["invoice_date"])

print(f"{len(tx):,} line items")
print(f"{tx['customer_id'].nunique():,} customers")
print(f"{tx['stock_code'].nunique():,} SKUs")
print(f"{tx['invoice_no'].nunique():,} invoices")
print(f"{tx['invoice_date'].min():%Y-%m-%d} to {tx['invoice_date'].max():%Y-%m-%d}")
print(f"£{tx['total_price'].sum():,.0f} revenue")
tx.head()

### Revenue over time

The Q4 peak is the signal every seasonal model is trying to capture. Note how few
of them there are — two years of weekly data gives at most two observations of an
annual cycle, which is why seasonal terms here are weakly identified.

In [ ]:
monthly = tx.set_index("invoice_date").resample("ME")["total_price"].sum()

fig, ax = plt.subplots()
ax.plot(monthly.index, monthly.values, marker="o", lw=2)
ax.set_title("Monthly revenue")
ax.set_ylabel("Revenue (£)")
ax.yaxis.set_major_formatter(lambda x, _: f"£{x:,.0f}")
plt.tight_layout()

## 2. Why MAPE is the wrong metric here

The previous version of this project reported "SARIMA: 37% MAPE" as its headline
result. Two things make that number untrustworthy on this data.

In [ ]:
panel = read_sql("SELECT stock_code, week, weekly_sales FROM ml_weekly_features")
panel["week"] = pd.to_datetime(panel["week"])

zero_share = (panel["weekly_sales"] == 0).mean()
print(f"{zero_share:.1%} of SKU-weeks have zero demand")
print(f"{(panel.groupby('stock_code')['weekly_sales'].min() == 0).mean():.1%} of SKUs "
      "have at least one zero-demand week")

**Problem one: MAPE divides by the actual.** Every one of those zero weeks is a
division by zero.

The old pipeline never hit this, but only by accident: grouping by week produced
rows solely for weeks that *had* a sale, so zero-demand weeks silently vanished.
That same bug also meant `shift(1)` computed "the previous week with a sale"
rather than "last week" — so the lag features were wrong too.

**Problem two: MAPE is asymmetric.** Over-forecasting is capped in how much it can
be punished; under-forecasting is not. Watch what happens to two forecasts that
are wrong by exactly the same number of units:

In [ ]:
from retail_intel.forecasting import metrics as M

actual = np.array([100.0] * 10)
over  = np.array([150.0] * 10)   # 50 units too high
under = np.array([50.0] * 10)    # 50 units too low

comparison = pd.DataFrame({
    "MAE":   [M.mae(actual, over),   M.mae(actual, under)],
    "MAPE":  [M.mape(actual, over),  M.mape(actual, under)],
    "sMAPE": [M.smape(actual, over), M.smape(actual, under)],
    "WAPE":  [M.wape(actual, over),  M.wape(actual, under)],
}, index=["over-forecast by 50", "under-forecast by 50"]).round(2)
comparison

Identical absolute error, and MAPE scores them the same here — but only because
the errors are symmetric around the actual. Push the under-forecast toward zero
and MAPE saturates at 100% while the over-forecast side runs to infinity. The
practical consequence is that optimising MAPE biases a model *downward*, and a
model that systematically under-forecasts causes stockouts.

For inventory that is exactly backwards: a stockout usually costs several times a
week of holding.

**MASE** avoids both problems. It scales absolute error by the in-sample error of
a seasonal-naive forecast, so it is defined at zero, symmetric, scale-free, and
has a meaning you can act on: **below 1 beats the benchmark.**

## 3. Baselines are not a formality

A percentage error means nothing on its own. The only question is whether the
model beats what a planner would do without it — and for weekly retail, that is
"same week last year".

In [ ]:
summary = pd.read_csv("../reports/backtest_summary.csv")
summary[["model", "n_skus", "mase_mean", "mase_median", "wape_mean",
         "bias_pct", "coverage_pct", "win_rate_vs_baseline"]].round(3)

Read the two columns that matter together:

- `mase_mean` below 1 means the model beats seasonal naive *on average*.
- `win_rate_vs_baseline` is the share of individual SKUs where it wins.

A model can look good on the first and poor on the second, which means a handful
of high-volume SKUs are carrying it. That is worth knowing before deploying it
across the catalogue.

In [ ]:
fig, ax = plt.subplots()
ordered = summary.sort_values("mase_mean")
colors = ["tab:green" if m < 1 else "tab:red" for m in ordered["mase_mean"]]
ax.barh(ordered["model"], ordered["mase_mean"], color=colors)
ax.axvline(1.0, color="black", ls="--", lw=1.5)
ax.text(1.01, -0.4, "seasonal naive", rotation=90, va="bottom", fontsize=9)
ax.set_xlabel("MASE (lower is better; < 1 beats the baseline)")
ax.set_title("Backtest leaderboard")
plt.tight_layout()

## 4. Bias: the failure a good error score hides

A model can have respectable absolute error while being consistently wrong in one
direction. Absolute error will not show it, but inventory will — the warehouse
either fills up or empties out, slowly.

In [ ]:
fig, ax = plt.subplots()
ordered = summary.sort_values("bias_pct")
ax.barh(ordered["model"], ordered["bias_pct"],
        color=["tab:red" if abs(b) > 15 else "tab:blue" for b in ordered["bias_pct"]])
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Forecast bias (%)  ·  positive = over-forecasts")
ax.set_title("Systematic bias by model")
plt.tight_layout()

print("Persistent over-forecasting quietly builds up stock; persistent")
print("under-forecasting quietly causes stockouts. Neither shows up in MAE.")

## 5. No single model wins everywhere

This is the argument for selecting a champion per SKU rather than declaring one
global winner. A steady high-volume SKU and an intermittent long-tail one are
different forecasting problems.

In [ ]:
champions = pd.read_csv("../reports/champions.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
champions["champion"].value_counts().plot.barh(ax=axes[0])
axes[0].set_title("Champion model by SKU count")
axes[0].set_xlabel("SKUs")

axes[1].hist(champions["improvement_pct"].dropna(), bins=25, edgecolor="white")
axes[1].axvline(0, color="red", ls="--", lw=1.5)
axes[1].set_title("Improvement over baseline, per SKU")
axes[1].set_xlabel("% reduction in MASE")
plt.tight_layout()

beat = (champions["champion_mase"] < champions["baseline_mase"]).mean()
print(f"{beat:.1%} of SKUs have a model that beats seasonal naive.")
print("The rest are served BY the baseline: shipping a complex model that loses")
print("to a one-line heuristic is worse than shipping the heuristic.")

## 6. Turning error into money

The step that makes any of this matter. Given holding cost `Co` and stockout cost
`Cu`, the newsvendor critical ratio gives the optimal service level directly —
it is not a policy choice once you know the costs:

$$SL^* = \\frac{C_u}{C_u + C_o}$$

Safety stock then follows from the **forecast error** distribution, not from
demand variance. Using demand variance is a common and expensive mistake: it
charges the model for seasonality it predicted perfectly well.

In [ ]:
from retail_intel.business import inventory as INV

for stockout, holding in [(2.50, 0.15), (10.0, 0.15), (2.50, 1.00)]:
    sl = INV.optimal_service_level(stockout, holding)
    ss = INV.safety_stock(demand_std=20, lead_time_weeks=2, service_level=sl)
    print(f"stockout £{stockout:5.2f}/unit, holding £{holding:.2f}/unit/week "
          f"→ service level {sl:.1%}, safety stock {ss:5.1f} units")

In [ ]:
# Price two forecasts of the same demand: one accurate, one baseline-quality.
rng = np.random.default_rng(0)
sku = panel.groupby("stock_code").size().idxmax()
series = panel.loc[panel["stock_code"] == sku, "weekly_sales"].reset_index(drop=True)

actual = series.iloc[-12:].to_numpy()
accurate = actual + rng.normal(0, actual.std() * 0.3, len(actual))
baseline = actual + rng.normal(0, actual.std() * 1.0, len(actual))

results = INV.compare_models(
    actual,
    {"champion": accurate, "seasonal_naive": baseline},
    {"champion": actual.std() * 0.3, "seasonal_naive": actual.std()},
)
pd.DataFrame([r.to_dict() for r in results])

In [ ]:
savings = INV.savings_vs_baseline(results)
print(f"Champion cost:  £{savings['champion_cost']:,.2f}")
print(f"Baseline cost:  £{savings['baseline_cost']:,.2f}")
print(f"Saving:         £{savings['absolute_saving']:,.2f}  "
      f"({savings['pct_saving']:.1f}%)")
print()
print("This is the sentence the forecasting work has to earn:")
print("not 'MASE improved by 0.2' but 'the same service level costs less'.")

## 7. Customers: what they did, versus what they will do

RFM segments describe the past. They cannot distinguish a customer who has churned
from one who is simply slow — which is the central question in retail, where
nobody cancels, they just stop coming back.

BG/NBD models that directly.

In [ ]:
segments = read_sql("SELECT * FROM customer_segments")

profile = (segments.groupby("segment_label")
           .agg(customers=("customer_id", "count"),
                avg_recency=("recency", "mean"),
                avg_frequency=("frequency", "mean"),
                avg_monetary=("monetary", "mean"),
                predicted_clv=("predicted_clv_90d", "mean"),
                churn_prob=("churn_probability", "mean"))
           .round(2)
           .sort_values("avg_monetary", ascending=False))
profile

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
scatter = ax.scatter(segments["churn_probability"], segments["predicted_clv_90d"],
                     c=segments["frequency"], cmap="viridis", alpha=0.65, s=28)
ax.axvline(0.5, color="red", ls="--", lw=1)
threshold = segments["predicted_clv_90d"].quantile(0.75)
ax.axhline(threshold, color="red", ls="--", lw=1)
ax.set_xlabel("Churn probability")
ax.set_ylabel("Predicted 90-day value (£)")
ax.set_title("Where retention budget belongs (top-right quadrant)")
plt.colorbar(scatter, label="Purchase frequency")
plt.tight_layout()

at_risk = segments[(segments["churn_probability"] > 0.5) &
                   (segments["predicted_clv_90d"] > threshold)]
print(f"{len(at_risk)} customers are both high-value and likely churned,")
print(f"carrying £{at_risk['predicted_clv_90d'].sum():,.0f} of predicted 90-day value.")
print()
print("Ranking by past spend alone would surface loyal customers instead,")
print("who need nothing.")

## 8. What is still wrong

Stated because they are real limitations, not because they are small:

- **No stockout data.** A week that sold nothing because the item was unavailable
  is recorded as zero demand. This is the largest source of bias in retail
  forecasting and nothing here addresses it.
- **No price or promotion features.** A demand spike caused by a discount is
  modelled as unexplained seasonality.
- **Two years of history** — at most two observations of any annual peak.
- **Top 100 SKUs only.** The long tail is intermittent demand, which wants
  Croston's method.
- **Uplift is validated against a simulation**, not measured: this data has no
  control group.

The most valuable next step is not a better model. It is getting stockout flags
and promotion calendars into the warehouse, which would improve accuracy more
than any amount of hyperparameter tuning.